# 🧪 Phase 2 Pipeline Test — AnyProjector
Kiểm tra từng bước trong pipeline Alignment Training.
Chạy từng cell để xem output mỗi khâu.

In [ ]:
!pip install -U ipykernel

In [ ]:
# Cell 1: Imports & Environment
import sys, os, json
import torch
import torchaudio
import soundfile as sf
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / 'src').exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# Cell 2: Load Dataset Metadata
DATASET_DIR = PROJECT_ROOT / "dataset" / "phase2_alignment"
METADATA_FILE = DATASET_DIR / "metadata.jsonl"

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = [json.loads(line) for line in f if line.strip()]

print(f"📊 Loaded {len(metadata)} entries from metadata.jsonl")
print(f"   Original: {sum(1 for m in metadata if m.get('augmented_from') is None)}")
print(f"   Augmented: {sum(1 for m in metadata if m.get('augmented_from') is not None)}")
print(f"\n--- Sample entry ---")
print(json.dumps(metadata[0], indent=2, ensure_ascii=False))

In [ ]:
# Cell 3: Load & Inspect Audio
sample = metadata[0]
audio_path = DATASET_DIR / sample["audio_file"]

# Use soundfile (avoids torchcodec DLL issues on Windows)
import numpy as np
audio_np, sr = sf.read(str(audio_path))
waveform = torch.from_numpy(audio_np.astype(np.float32)).unsqueeze(0)  # (1, samples)

print(f"🎤 Audio: {sample['audio_file']}")
print(f"   Transcript: {sample['transcript']}")
print(f"   Shape: {waveform.shape}  (channels, samples)")
print(f"   Sample rate: {sr} Hz")
print(f"   Duration: {waveform.shape[1] / sr:.2f}s")

# Resample to 16kHz if needed
TARGET_SR = 16000
if sr != TARGET_SR:
    resampler = torchaudio.transforms.Resample(sr, TARGET_SR)
    waveform = resampler(waveform)
    print(f"   Resampled: {waveform.shape} @ {TARGET_SR}Hz")

In [ ]:
# Cell 4: Load Whisper Encoder (frozen)
from transformers import WhisperModel, WhisperProcessor

ENCODER_ID = "openai/whisper-medium"
print(f"🔊 Loading encoder: {ENCODER_ID}")

processor = WhisperProcessor.from_pretrained(ENCODER_ID)
whisper_full = WhisperModel.from_pretrained(ENCODER_ID)
encoder = whisper_full.encoder.to(DEVICE).eval()
del whisper_full  # Free decoder memory

# Freeze
for p in encoder.parameters():
    p.requires_grad = False

encoder_dim = encoder.config.d_model
total_params = sum(p.numel() for p in encoder.parameters())
print(f"   encoder_dim (d_model): {encoder_dim}")
print(f"   Parameters: {total_params:,} (all frozen)")

In [ ]:
# Cell 5: Audio → Encoder Output
# Process audio through Whisper's feature extractor → mel spectrogram → encoder
input_features = processor(
    waveform.squeeze().numpy(),
    sampling_rate=TARGET_SR,
    return_tensors="pt"
).input_features.to(DEVICE)

print(f"📥 Mel spectrogram shape: {input_features.shape}")

with torch.no_grad():
    encoder_output = encoder(input_features).last_hidden_state

print(f"📤 Encoder output shape: {encoder_output.shape}")
print(f"   = (batch={encoder_output.shape[0]}, seq_len={encoder_output.shape[1]}, encoder_dim={encoder_output.shape[2]})")

In [ ]:
# Cell 6: Create Projector (trainable)
from src.projector import AnyProjector
from transformers import AutoConfig

# Auto-detect LLM hidden_size (chỉ tải config.json, không tải weights)
LLM_ID = "google/gemma-4-E2B-it"  # ← Đổi model ở đây, Cell 8 sẽ dùng cùng ID
llm_config = AutoConfig.from_pretrained(LLM_ID)
LLM_DIM = llm_config.hidden_size
print(f"🔍 Auto-detected LLM dim from {LLM_ID}: {LLM_DIM}")

projector = AnyProjector(encoder_dim=encoder_dim, llm_dim=LLM_DIM).to(DEVICE)
projector.train()

print(f"🔗 {projector}")
print(f"\n--- Gradient check ---")
for name, p in projector.named_parameters():
    print(f"   {name}: requires_grad={p.requires_grad}, shape={list(p.shape)}")

In [ ]:
# Cell 7: Projector Forward
audio_embeds = projector(encoder_output)

print(f"📥 Input:  {encoder_output.shape}  (batch, {encoder_output.shape[1]}, encoder_dim={encoder_dim})")
print(f"📤 Output: {audio_embeds.shape}  (batch, {audio_embeds.shape[1]}, llm_dim={LLM_DIM})")
print(f"   Temporal compression: {encoder_output.shape[1]} → {audio_embeds.shape[1]} tokens (÷2)")

In [ ]:
# Cell 8: Load LLM + Tokenizer (4-bit on GPU)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# LLM_ID đã được set ở Cell 6 (đổi ở đó, không đổi ở đây)
print(f"🧠 Loading LLM: {LLM_ID}")

tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    print("   Using 4-bit NF4 quantization (GPU)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    try:
        # Try GPU-only first
        llm = AutoModelForCausalLM.from_pretrained(
            LLM_ID, quantization_config=bnb_config,
            device_map="auto", torch_dtype=torch.float16,
        )
    except ValueError as e:
        if "CPU or the disk" in str(e):
            # VRAM không đủ → cho phép offload một phần sang CPU
            print("   ⚠️ VRAM không đủ, cho phép CPU offload...")
            bnb_config.llm_int8_enable_fp32_cpu_offload = True
            llm = AutoModelForCausalLM.from_pretrained(
                LLM_ID, quantization_config=bnb_config,
                device_map="auto", torch_dtype=torch.float16,
            )
        else:
            raise
else:
    print("   No GPU — loading float32 on CPU (slow)")
    llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float32).to(DEVICE)

# Freeze
for p in llm.parameters():
    p.requires_grad = False
llm.eval()

actual_llm_dim = llm.config.hidden_size
total_llm_params = sum(p.numel() for p in llm.parameters())
print(f"   hidden_size: {actual_llm_dim}")
print(f"   Parameters: {total_llm_params:,} (all frozen)")
print(f"   Vocab size: {llm.config.vocab_size}")

assert actual_llm_dim == LLM_DIM, f"LLM dim mismatch! Expected {LLM_DIM}, got {actual_llm_dim}"
print(f"   ✅ LLM dim matches Projector output!")

In [ ]:
# Cell 9: Create Text Prompt Embeddings
PROMPT_TEXT = "Phiên âm đoạn audio sau bằng tiếng Việt:"
prompt_tokens = tokenizer(PROMPT_TEXT, return_tensors="pt").input_ids.to(DEVICE)

embed_layer = llm.get_input_embeddings()
with torch.no_grad():
    prompt_embeds = embed_layer(prompt_tokens)

print(f"📝 Prompt: \"{PROMPT_TEXT}\"")
print(f"   Token IDs: {prompt_tokens.shape} → {prompt_tokens[0].tolist()}")
print(f"   Prompt embeddings: {prompt_embeds.shape}")
print(f"   = (batch=1, {prompt_embeds.shape[1]} prompt tokens, llm_dim={LLM_DIM})")

In [ ]:
# Cell 10: Combine & LLM Forward
# Concatenate: [prompt_embeds | audio_embeds]
combined = torch.cat([prompt_embeds, audio_embeds], dim=1)
print(f"🔗 Combined input:")
print(f"   prompt_embeds:  {prompt_embeds.shape[1]} tokens")
print(f"   audio_embeds:   {audio_embeds.shape[1]} tokens")
print(f"   combined:       {combined.shape[1]} tokens")
print(f"   combined shape: {combined.shape}")

# Forward through LLM
with torch.no_grad():
    llm_output = llm(inputs_embeds=combined)
    logits = llm_output.logits

print(f"\n📤 LLM output logits: {logits.shape}")
print(f"   = (batch=1, seq_len={logits.shape[1]}, vocab_size={logits.shape[2]})")

In [ ]:
# Cell 11: Calculate Loss
# Target = transcript tokens
transcript = sample["transcript"]
target_tokens = tokenizer(
    transcript,
    return_tensors="pt",
    padding=False,
    add_special_tokens=False,
).input_ids.to(DEVICE)

print(f"🎯 Target transcript: \"{transcript}\"")
print(f"   Target token IDs shape: {target_tokens.shape}")

# The loss is computed on the LAST N tokens of the LLM output,
# where N = len(target_tokens). We shift logits by 1 for autoregressive prediction.
n_target = target_tokens.shape[1]
n_combined = combined.shape[1]

# We need combined to be long enough. Typically we'd concatenate target embeds too
# for teacher forcing. Let's do it properly:
target_embeds = embed_layer(target_tokens)
full_input = torch.cat([prompt_embeds, audio_embeds, target_embeds], dim=1)

print(f"\n--- Teacher Forcing Setup ---")
print(f"   full_input: prompt({prompt_embeds.shape[1]}) + audio({audio_embeds.shape[1]}) + target({target_embeds.shape[1]}) = {full_input.shape[1]} tokens")

# Forward with full input (teacher forcing)
outputs = llm(inputs_embeds=full_input)
logits = outputs.logits  # (1, total_seq_len, vocab_size)

# Loss: predict target tokens from the positions after audio
# Shift: logits[..., audio_end:-1, :] predicts target_tokens[..., :]
audio_end = prompt_embeds.shape[1] + audio_embeds.shape[1]
predict_logits = logits[:, audio_end - 1 : audio_end - 1 + n_target, :]

loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(
    predict_logits.reshape(-1, logits.shape[-1]),
    target_tokens.reshape(-1),
)

print(f"\n📉 Loss = {loss.item():.4f}")
print(f"   (random baseline ≈ {torch.log(torch.tensor(float(llm.config.vocab_size))).item():.2f})")

In [ ]:
# Cell 12: Backward — Verify Gradients
loss.backward()

print("🔙 Backward pass completed!\n")
print("--- Projector gradients ---")
for name, p in projector.named_parameters():
    grad_norm = p.grad.norm().item() if p.grad is not None else 0
    print(f"   {name}: grad_norm={grad_norm:.6f}")

print("\n--- Encoder gradients (should be None) ---")
has_grad = any(p.grad is not None for p in encoder.parameters())
print(f"   Any encoder param has grad? {has_grad} ({'❌ BUG!' if has_grad else '✅ Correct'})")

print("\n--- LLM gradients (should be None) ---")
has_grad = any(p.grad is not None for p in llm.parameters())
print(f"   Any LLM param has grad? {has_grad} ({'❌ BUG!' if has_grad else '✅ Correct'})")

In [ ]:
# Cell 13: Mini Training Loop (3 steps)
projector.zero_grad()
optimizer = torch.optim.AdamW(projector.parameters(), lr=1e-4, weight_decay=0.01)

print("🏋️ Mini training loop (3 steps, 1 sample):\n")
for step in range(1, 4):
    optimizer.zero_grad()

    # Forward pipeline
    with torch.no_grad():
        enc_out = encoder(input_features).last_hidden_state
    
    audio_emb = projector(enc_out)
    
    with torch.no_grad():
        p_emb = embed_layer(prompt_tokens)
        t_emb = embed_layer(target_tokens)
    
    full_in = torch.cat([p_emb, audio_emb, t_emb], dim=1)
    out = llm(inputs_embeds=full_in)
    
    pred_logits = out.logits[:, audio_end - 1 : audio_end - 1 + n_target, :]
    step_loss = loss_fn(pred_logits.reshape(-1, out.logits.shape[-1]), target_tokens.reshape(-1))
    
    step_loss.backward()
    torch.nn.utils.clip_grad_norm_(projector.parameters(), max_norm=1.0)
    optimizer.step()
    
    print(f"   Step {step}: loss = {step_loss.item():.4f}")

print("\n✅ Pipeline Phase 2 hoạt động! Projector nhận gradient, Encoder + LLM frozen.")
print("   Tiếp theo: Scale lên full dataset + nhiều epoch để loss hội tụ.")